# HVAC 2.0 — Stage 1: Domain-Shift Quantification (frozen evaluation, no training)

Corresponds to `proposal/proposal.md` v6, §7, Stage 1. **All 5 pretrained models are frozen; no parameters are updated.** Evaluation only.

| Evaluation | Models | Evaluated on | Groups |
| --- | --- | --- | --- |
| **Small shift** | 4 single-ID models only | The other 3 unseen MIMII machine IDs (each with its own 160 normal + 320 abnormal) | 12 |
| **Large shift** | All 5 | 180 self-recorded fan clips, 10-fold LORO | 5 |

**Metrics recorded**: AUC, best F1, median reconstruction MSE on normal samples, threshold shift ratio.

**This stage produces no new models**; only `logs/` is created under `Stage1/`.

---

### Why frozen evaluation still uses LORO

The models are not trained, so a clip's anomaly score is **completely independent** of which fold it falls in — scoring all 180 clips once is enough,
and the fold split is only a **post-hoc aggregation**. Results are still aggregated over exactly the same 10 folds as Stage 2 so that
Stage 1 baseline numbers can be **directly subtracted** from Stage 2 fine-tuned numbers to obtain the improvement; with different aggregation the two would not be comparable.

Stage 2 is different: there each fold is a separately fine-tuned model (10 models), so pooled and per-fold give genuinely different numbers.

---

**How to run**: `Runtime → Run all`. **Runtime**: T4 GPU (CPU also works; this stage is computationally light).

**Embedded config** sha256 = `8430e581ae6b4e14...`, the same config used in Stage 0;
it is compared against the value recorded in `stage0_splits_seed42.json`.

> Note: `8430e581ae6b4e14...` is the hash of the original config file. Translating the config's comments changes the hash,
> so the sha256 mismatch warning in section 5 is expected and harmless (splits are read from file, not recomputed).

## 1 · Environment and dependencies

In [ ]:
import sys, os, re, time, json, glob, hashlib, zipfile, subprocess
from collections import defaultdict

def _ensure(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("pyyaml", "yaml")
_ensure("librosa")
_ensure("tqdm")

import numpy as np
import pandas as pd
import yaml
import librosa
import torch
import torch.nn as nn
from sklearn import metrics
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("librosa :", librosa.__version__)
print("torch   :", torch.__version__)
print("Device  :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU (this stage is light; CPU is acceptable)")

## 2 · Path constants ← edit paths here

All reads and writes go through the constants below; no hard-coded paths appear anywhere else.

- **Read**: `BASELINE_DIR` (Stage 0 outputs, read-only)
- **Write**: `STAGE1_DIR` (this stage's outputs)
- **Local disk**: extraction and feature cache, cleared on runtime restart

In [ ]:
# ============================ EDIT ===============================
DRIVE_ROOT    = "/content/drive/MyDrive/HVAC_fan"

MIMII_ZIP     = f"{DRIVE_ROOT}/0_dB_fan.zip"          # source-domain zip
TARGET_ZIP    = f"{DRIVE_ROOT}/raw_audio.zip"         # target-domain zip (180 self-recorded clips)

RESULTS_ROOT  = f"{DRIVE_ROOT}/HVAC2_Results"
BASELINE_DIR  = f"{RESULTS_ROOT}/Baseline"            # Stage 0 outputs (read)
STAGE1_DIR    = f"{RESULTS_ROOT}/Stage1"              # this stage's outputs (write)

EXTRACT_ROOT  = "/content/data"                        # local disk: extraction + feature cache
# ===============================================================

BASELINE_MODELS = f"{BASELINE_DIR}/models"
BASELINE_LOGS   = f"{BASELINE_DIR}/logs"
STAGE1_LOGS     = f"{STAGE1_DIR}/logs"                 # no models produced in this stage, so no models/

MIMII_EXTRACT   = f"{EXTRACT_ROOT}/mimii"
MIMII_ROOT      = f"{MIMII_EXTRACT}/fan"               # contains id_00/id_02/id_04/id_06
TARGET_EXTRACT  = f"{EXTRACT_ROOT}/target"
TARGET_ROOT     = f"{TARGET_EXTRACT}/raw_audio"        # contains {condition}/{voltage}/{noise}/
FEATURE_DIR     = f"{EXTRACT_ROOT}/features"           # same cache dir as Stage 0, reusable

for label, p in [("Source zip", MIMII_ZIP), ("Target zip", TARGET_ZIP),
                 ("Stage 0 outputs", BASELINE_DIR), ("Stage 1 outputs", STAGE1_DIR),
                 ("Feature cache", FEATURE_DIR)]:
    print(f"  {label:<16} {p}")

## 3 · Config (`code/configs/config.yaml` embedded verbatim) + self-check

Exactly the same config and self-check as Stage 0.

In [ ]:
_CONFIG_YAML_RAW = r"""
# ============================================================================
# HVAC 2.0 — experiment configuration (single source of truth)
# Matches proposal/proposal.md v6. On any conflict, the proposal wins and this file is updated.
# This file holds only "constants and rules"; no training logic.
# ============================================================================

project: HVAC_2.0
proposal_version: v6
seed: 42                      # the one random seed for the whole project (proposal §4.1)

# ---------------------------------------------------------------------------
# Paths. env selects which block is used; keep colab when running on Colab, set local for local debugging.
# ---------------------------------------------------------------------------
paths:
  env: colab                  # colab | local

  colab:
    drive_root: /content/drive/MyDrive
    # Source and target domains are both zips on Drive. Always extract to the Colab local disk /content/data,
    # never back onto Drive: Drive small-file IO is extremely slow, and it would push thousands of wavs to the cloud.
    # The local disk is ephemeral and must be re-extracted after a runtime restart -- extraction should be idempotent (skip if the directory exists).
    extract_root: /content/data

    # Source domain: MIMII 0dB fan
    mimii_zip: /content/drive/MyDrive/HVAC_fan/0_dB_fan.zip
    mimii_extract_to: /content/data/mimii
    mimii_root: /content/data/mimii/fan          # contains id_00/id_02/id_04/id_06

    # Target domain: 180 self-recorded fan clips
    target_zip: /content/drive/MyDrive/HVAC_fan/raw_audio.zip
    target_extract_to: /content/data/target
    target_root: /content/data/target/raw_audio  # contains {condition}/{voltage}/{noise}/
    recording_log: null                          # inventory check only; metadata comes from filenames

    # Outputs: written back to Drive so they survive runtime restarts
    outputs_root: /content/drive/MyDrive/HVAC_fan/HVAC2_Results

  local:
    drive_root: null
    extract_root: null
    # Already extracted locally; no zips needed
    mimii_zip: null
    mimii_extract_to: null
    mimii_root: data/mimii/fan
    target_zip: null
    target_extract_to: null
    target_root: data/raw_audio
    recording_log: data/metadata/recording_log.csv
    outputs_root: outputs

outputs:
  # Stages 2+3 total ~1032 train/eval runs. Weights are discarded after use; only CSVs are saved (proposal §7, stage 2)
  persist_finetuned_weights: false
  pretrained_models_dir: models        # relative to outputs_root; these 5 weights must be kept
  results_dir: logs
  figures_dir: figures

# ---------------------------------------------------------------------------
# Source domain: MIMII fan 0dB (proposal §4.1)
# ---------------------------------------------------------------------------
source_domain:
  name: MIMII_fan_0dB
  machine_ids: [id_00, id_02, id_04, id_06]

  # Official counts, used for a startup self-check: scanned file counts must match exactly, else fail immediately
  official_counts:
    id_00: {normal: 1011, abnormal: 407}
    id_02: {normal: 1016, abnormal: 359}
    id_04: {normal: 1033, abnormal: 348}
    id_06: {normal: 1015, abnormal: 361}

  # Evaluation set: class ratio 1:2, aligned with the target domain's 60:120 (proposal §3)
  eval_set:
    n_normal: 160
    n_abnormal: 320

  # Training pool = all normal clips of that machine ID − the 160 used by the evaluation set
  train_pool_size:
    id_00: 851
    id_02: 856
    id_04: 873
    id_06: 855

  # Merged model: equal contribution from each machine ID
  merged_model:
    # Evaluation set must be a subset of each ID's fixed evaluation set; no independent re-draw (proposal §4.1)
    eval_per_id: {normal: 40, abnormal: 80}
    # Training set: 215 clips drawn from each ID's "training pool", so the total matches the single-ID models
    train_per_id: 215
    train_total: 860

# ---------------------------------------------------------------------------
# Target domain: 180 self-recorded fan clips (proposal §4.2)
# ---------------------------------------------------------------------------
target_domain:
  name: self_recorded_fan
  device: Razer Seiren V3 Mini

  conditions: [normal, blocked, imbalance]
  normal_conditions: [normal]                 # 60 clips, the only part that may enter training
  abnormal_conditions: [blocked, imbalance]   # 120 clips, evaluation only
  voltages: ["4V", "8V", "12V"]
  noise_levels: [quiet, noise]
  n_runs: 10

  n_clips_total: 180
  n_normal_total: 60
  n_abnormal_total: 120

  # Filenames are the single source of truth for metadata; recording_log.csv is only an inventory check
  # (the csv has 206 rows with duplicates and misaligned columns; after dedup, the 180 filenames form a complete 18×10 grid)
  filename_pattern: "{condition}_{voltage}_{noise}_run{run:02d}.wav"
  relpath_pattern: "{condition}/{voltage}/{noise}/{filename}"

# ---------------------------------------------------------------------------
# Feature extraction (proposal §5, identical to baseline_runner.ipynb)
# ---------------------------------------------------------------------------
features:
  sr: 16000
  duration_sec: 10
  n_mels: 64
  frames: 5           # concatenate 5 adjacent frames
  n_fft: 1024
  hop_length: 512
  power: 2.0
  input_dim: 320      # = n_mels * frames
  # Each 10 s clip yields ~309 vectors of dim 320 (313 frames − 5 + 1)

# ---------------------------------------------------------------------------
# Model architecture (proposal §5)
# ---------------------------------------------------------------------------
model:
  arch: MIMII_Baseline_AE
  encoder_dims: [320, 64, 64, 8]
  decoder_dims: [8, 64, 64, 320]
  bottleneck_dim: 8
  activation: relu
  output_activation: none      # official baseline has no activation on the output layer
  # Layer indices inside the torch module, used by the layer schemes to freeze/unfreeze by name
  layer_index:
    encoder_linears: ["encoder.0", "encoder.2", "encoder.4"]   # 320→64, 64→64, 64→8
    decoder_linears: ["decoder.0", "decoder.2", "decoder.4"]   # 8→64, 64→64, 64→320

# ---------------------------------------------------------------------------
# Pretraining (stage 0, proposal §5 / §7 stage 0)
# ---------------------------------------------------------------------------
pretrain:
  optimizer: adam
  lr: 0.001            # torch Adam default, same as baseline_runner.ipynb
  loss: mse
  batch_size: 512
  epochs: 50
  validation_split: 0.1
  shuffle: true
  # Right after training, evaluate once on the model's own 480-clip source eval set and store as the "source baseline" (subtrahend for forgetting)
  record_source_baseline: true

# ---------------------------------------------------------------------------
# Fine-tuning (stage 2) — all four layer schemes share one hyperparameter set, not tuned per scheme (proposal §7, stage 2)
# ---------------------------------------------------------------------------
finetune:
  optimizer: adam
  loss: mse
  lr: 0.0001           # 1/10 of pretraining lr; at the 40-clip full level the movement budget is still < 2.5% of pretraining
  epochs: 30           # derived from a step budget: ~750 updates at the 5-clip level, ~5820 at the 40-clip level
  batch_size: 64       # 512 would give only 4 batches per epoch at the 5-clip level, too few for Adam momentum to warm up
  source_replay: false          # no source-domain replay (proposal §7, stage 2)
  early_stopping: false         # no early stopping; rationale in proposal §7, stage 2
  restart_from_pretrained_each_fold: true   # every LORO fold must restart from the pretrained weights

# ---------------------------------------------------------------------------
# LORO validation protocol (proposal §6)
# ---------------------------------------------------------------------------
loro:
  applies_to: large_shift_only     # no LORO for small shift (proposal §6.4)
  n_folds: 10                      # fold k holds out the whole run == k+1
  fold_key: run                    # determined by runNN in the filename
  clips_per_fold: 18               # one clip from each of the 18 conditions
  normals_per_fold: 6
  abnormals_per_fold: 12
  max_train_normals: 54            # 60 − 6, which is why the data levels stop at 40
  report: [pooled, per_fold]       # compute both views (proposal §6.2)
  per_fold_stats: [mean, std]
  pooled_normalization: none       # no per-fold normalization; listed as a limitation

# ---------------------------------------------------------------------------
# Experiment grid (proposal §7, stage 2)
# ---------------------------------------------------------------------------
grid:
  data_levels: [5, 10, 20, 40]

  # Nested sampling: 5 ⊂ 10 ⊂ 20 ⊂ 40 (proposal §4.3)
  nested_sampling:
    enabled: true
    rule: "Candidates are put in a fixed order by a fixed rule; level n takes the first n"
    # Ordering = sort by filename for a deterministic initial order, then shuffle with a numpy RNG
    canonical_order: filename_sort_then_shuffle
    rng: numpy_default_rng
    # Small shift: order each target ID's training pool once (seed scope includes only the ID)
    small_shift_seed_scope: [seed, target_id]
    # Large shift: available clips differ per fold, so each fold is ordered separately (seed scope includes the fold)
    large_shift_seed_scope: [seed, fold]

  layer_schemes:
    full:
      desc: Full fine-tuning
      trainable: ["encoder.0", "encoder.2", "encoder.4", "decoder.0", "decoder.2", "decoder.4"]
    decoder_only:
      desc: Decoder only
      trainable: ["decoder.0", "decoder.2", "decoder.4"]
    encoder_only:
      desc: Encoder only
      trainable: ["encoder.0", "encoder.2", "encoder.4"]
    bottleneck_only:
      desc: Bottleneck-adjacent layers only (the two layers into and out of the bottleneck, 64→8 and 8→64)
      trainable: ["encoder.4", "decoder.0"]

# ---------------------------------------------------------------------------
# Experiment enumeration per stage (counts are self-checked at startup by config.py; mismatch raises)
# ---------------------------------------------------------------------------
stages:
  stage0_pretrain:
    n_models: 5                  # 4 single-ID + 1 merged
  stage1_frozen_eval:
    small_shift_pairs: 12        # 4 starts × 3 unseen IDs each
    large_shift_models: 5        # all 5 models evaluated on large shift (incl. merged)
    large_shift_folds: 10
    shift_ratio:
      aggregation: median
      denominator: source_eval_normals          # the 160 normals of that ID's eval set
      numerator_small_shift: target_eval_normals # the 160 normals of the target ID's eval set
      numerator_large_shift: all_60_normals      # frozen eval has no leakage, so use all 60 clips
  stage2_finetune:
    small_shift:
      starts: 4
      targets_per_start: 3
      folds: 1
      n_configs: 192
      n_runs: 192
    large_shift:
      starts: 5                  # 4 single-ID + 1 merged
      targets_per_start: 1
      folds: 10
      n_configs: 80
      n_runs: 800
    n_configs_total: 272
    n_runs_total: 992
  stage3_no_pretrain:
    init: random
    layer_scheme: full           # random init has no "learned layers", so no layer-scheme variable
    n_configs: 4
    folds: 10
    n_runs: 40
    source_metrics: null         # no source domain; reported as N/A

total_runs: 1032                 # 992 + 40

# ---------------------------------------------------------------------------
# Evaluation metrics (proposal §3)
# ---------------------------------------------------------------------------
metrics:
  primary: auc                   # floor is always 0.5, independent of class ratio
  secondary: best_f1             # max over a scan of all thresholds
  best_f1_trivial_floor: 0.8     # at a 1:2 class ratio, "predict all abnormal" already gives 0.8
  n_threshold_scan_points: 1000  # matches the linspace in baseline_runner.ipynb
  clip_score: mean_mse_over_frames   # per clip = mean reconstruction MSE over all its feature vectors
  reconstruction_mse_aggregation: median   # threshold shift ratio uses the median
"""
CONFIG_YAML = _CONFIG_YAML_RAW.lstrip("\n")     # drop the newline after r""" so it matches the repository file byte-for-byte

CONFIG_SHA256 = hashlib.sha256(CONFIG_YAML.encode("utf-8")).hexdigest()
cfg = yaml.safe_load(CONFIG_YAML)


def rng_for(seed, *parts):
    import zlib
    tag = "|".join(str(p) for p in parts).encode("utf-8")
    return np.random.default_rng([seed, zlib.crc32(tag)])


def self_check(cfg):
    src, tgt, grid, stages = cfg["source_domain"], cfg["target_domain"], cfg["grid"], cfg["stages"]

    n_eval_normal = src["eval_set"]["n_normal"]
    n_eval_abnormal = src["eval_set"]["n_abnormal"]
    for mid in src["machine_ids"]:
        counts = src["official_counts"][mid]
        assert src["train_pool_size"][mid] == counts["normal"] - n_eval_normal
        assert counts["abnormal"] >= n_eval_abnormal

    merged, n_ids = src["merged_model"], len(src["machine_ids"])
    assert merged["eval_per_id"]["normal"] * n_ids == n_eval_normal
    assert merged["eval_per_id"]["abnormal"] * n_ids == n_eval_abnormal
    assert merged["train_per_id"] * n_ids == merged["train_total"]
    pools = list(src["train_pool_size"].values())
    assert min(pools) <= merged["train_total"] <= max(pools)

    src_ratio = n_eval_abnormal / n_eval_normal
    tgt_ratio = tgt["n_abnormal_total"] / tgt["n_normal_total"]
    assert abs(src_ratio - tgt_ratio) < 1e-9, "Source and target class ratios differ; the best-F1 floor would differ"
    floor = 2 * src_ratio / (src_ratio + 1) / (1 + src_ratio / (src_ratio + 1))
    assert abs(floor - cfg["metrics"]["best_f1_trivial_floor"]) < 1e-9

    n_cells = len(tgt["conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_cells * tgt["n_runs"] == tgt["n_clips_total"] == 180
    n_normal_cells = len(tgt["normal_conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_normal_cells * tgt["n_runs"] == tgt["n_normal_total"]

    feat = cfg["features"]
    assert feat["n_mels"] * feat["frames"] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][0] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][-1] == cfg["model"]["bottleneck_dim"]

    loro = cfg["loro"]
    assert loro["n_folds"] == tgt["n_runs"]
    assert loro["clips_per_fold"] == n_cells
    assert loro["normals_per_fold"] == n_normal_cells
    assert loro["normals_per_fold"] + loro["abnormals_per_fold"] == loro["clips_per_fold"]
    assert loro["max_train_normals"] == tgt["n_normal_total"] - loro["normals_per_fold"]

    s1 = stages["stage1_frozen_eval"]
    assert s1["small_shift_pairs"] == len(src["machine_ids"]) * (len(src["machine_ids"]) - 1)
    assert s1["large_shift_models"] == stages["stage0_pretrain"]["n_models"]
    assert s1["large_shift_folds"] == loro["n_folds"]


self_check(cfg)

SEED         = cfg["seed"]
MACHINE_IDS  = cfg["source_domain"]["machine_ids"]
ALL_MODELS   = MACHINE_IDS + ["merged"]
FEAT         = cfg["features"]
LORO         = cfg["loro"]
TGT          = cfg["target_domain"]
N_SCAN       = cfg["metrics"]["n_threshold_scan_points"]
F1_FLOOR     = cfg["metrics"]["best_f1_trivial_floor"]
S1           = cfg["stages"]["stage1_frozen_eval"]

print(f"Config loaded (proposal {cfg['proposal_version']}), sha256 = {CONFIG_SHA256[:16]}...")
print(f"Self-check passed ✅")
print(f"  Small-shift pairs    : {S1['small_shift_pairs']}  (4 start models × 3 unseen IDs each)")
print(f"  Large-shift models   : {S1['large_shift_models']}  × LORO {S1['large_shift_folds']} folds")
print(f"  Best-F1 trivial floor: {F1_FLOOR}")

## 4 · Mount Drive, unzip, verify directory layout

Both zips are extracted to local disk, **never written back to Drive**, idempotently (skipped if already in place).

Checks are as strict as in Stage 0: if the layout or file counts don't match, **the actual structure is printed and an error is raised**; it never continues silently.
The target domain is additionally checked for a complete 18×10 grid (3 conditions × 3 voltages × 2 environments, 10 runs per cell).

In [ ]:
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

os.makedirs(STAGE1_LOGS, exist_ok=True)


def print_tree(base, max_depth=3, max_entries=12):
    base = os.path.abspath(base)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        n_wav = sum(1 for f in files if f.lower().endswith(".wav"))
        print("  " * depth + (os.path.basename(root) or root) + "/" + (f"   [{n_wav} wav]" if n_wav else ""))
        dirs.sort()
        if len(dirs) > max_entries:
            print("  " * depth + f"  ... {len(dirs)} subdirectories, showing first {max_entries}")
            dirs[:] = dirs[:max_entries]


def unzip_idempotent(zip_path, extract_to, ready_fn, what):
    if not os.path.exists(zip_path):
        parent = os.path.dirname(zip_path)
        listing = sorted(os.listdir(parent)) if os.path.isdir(parent) else ["(directory does not exist)"]
        raise FileNotFoundError(
            f"❌ {what} zip not found: {zip_path}\n   Actual contents of {parent}:\n     "
            + "\n     ".join(listing)
            + f"\n   → If the path differs, edit the EDIT block in section 2 of this notebook."
        )
    if ready_fn():
        print(f"✅ {what} already on local disk, skipping unzip")
        return
    os.makedirs(extract_to, exist_ok=True)
    gb = os.path.getsize(zip_path) / 1024**3
    print(f"Unzipping {what} ({gb:.2f} GB) → {extract_to} ...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path) as zf:
        for m in tqdm(zf.infolist(), desc=f"unzip {what}", unit="file"):
            zf.extract(m, extract_to)
    print(f"  Done in {time.time() - t0:.0f} s")


# ---- Source domain ----
unzip_idempotent(
    MIMII_ZIP, MIMII_EXTRACT,
    lambda: all(os.path.isdir(os.path.join(MIMII_ROOT, m, l))
                for m in MACHINE_IDS for l in ("normal", "abnormal")),
    "source-domain MIMII")


def detect_dir(base, required_subdirs, leaf_subdirs=()):
    for root, dirs, _ in os.walk(base):
        if all(os.path.isdir(os.path.join(root, s)) for s in required_subdirs) and \
           all(os.path.isdir(os.path.join(root, s, l)) for s in required_subdirs for l in leaf_subdirs):
            return root
    return None


detected = detect_dir(MIMII_EXTRACT, MACHINE_IDS, ("normal", "abnormal"))
if detected is None or os.path.realpath(detected) != os.path.realpath(MIMII_ROOT):
    print(f"❌ Source-domain directory layout mismatch. Expected {MIMII_ROOT}, detected {detected}")
    print_tree(MIMII_EXTRACT)
    raise RuntimeError("Source-domain path problem — set MIMII_ROOT to the detected value above (EDIT block in section 2) and rerun this cell.")

mimii_counts = {m: {l: len(glob.glob(os.path.join(MIMII_ROOT, m, l, "*.wav")))
                    for l in ("normal", "abnormal")} for m in MACHINE_IDS}
bad = [(m, l, cfg["source_domain"]["official_counts"][m][l], mimii_counts[m][l])
       for m in MACHINE_IDS for l in ("normal", "abnormal")
       if mimii_counts[m][l] != cfg["source_domain"]["official_counts"][m][l]]
if bad:
    for m, l, e, g in bad:
        print(f"  {m}/{l}: expected {e}, got {g}")
    raise RuntimeError("Source-domain file counts do not match config — data problem.")
print(f"✅ Source domain verified: {MIMII_ROOT} ({sum(v[l] for v in mimii_counts.values() for l in v)} wav)")

# ---- Target domain ----
unzip_idempotent(
    TARGET_ZIP, TARGET_EXTRACT,
    lambda: all(os.path.isdir(os.path.join(TARGET_ROOT, c)) for c in TGT["conditions"]),
    "target-domain self-recordings")

detected_t = detect_dir(TARGET_EXTRACT, TGT["conditions"])
if detected_t is None or os.path.realpath(detected_t) != os.path.realpath(TARGET_ROOT):
    print(f"❌ Target-domain directory layout mismatch. Expected {TARGET_ROOT}, detected {detected_t}")
    print_tree(TARGET_EXTRACT)
    raise RuntimeError("Target-domain path problem — set TARGET_ROOT to the detected value above (EDIT block in section 2) and rerun this cell.")

CLIP_RE = re.compile(r"^(?P<condition>\w+?)_(?P<voltage>\d+V)_(?P<noise>\w+?)_run(?P<run>\d{2})\.wav$")


def parse_clip(path):
    """The filename is the single source of truth for target-domain metadata."""
    m = CLIP_RE.match(os.path.basename(path))
    if not m:
        raise ValueError(f"Filename does not follow the naming convention: {path}")
    d = m.groupdict()
    d["run"] = int(d["run"])
    return d


target_files = sorted(glob.glob(os.path.join(TARGET_ROOT, "**", "*.wav"), recursive=True))
if len(target_files) != TGT["n_clips_total"]:
    print(f"❌ Target domain should have {TGT['n_clips_total']} clips, found {len(target_files)}")
    print_tree(TARGET_EXTRACT)
    raise RuntimeError("Target-domain file count mismatch — data problem.")

grid = defaultdict(list)
for f in target_files:
    d = parse_clip(f)
    grid[(d["condition"], d["voltage"], d["noise"])].append(d["run"])
missing = [(k, sorted(v)) for k, v in grid.items() if sorted(v) != list(range(1, TGT["n_runs"] + 1))]
if len(grid) != LORO["clips_per_fold"] or missing:
    print(f"❌ 18×10 grid incomplete: {len(grid)} condition combinations (expected {LORO['clips_per_fold']})")
    for k, v in missing[:10]:
        print(f"   runs for {k}: {v}")
    raise RuntimeError("Target-domain grid incomplete — data problem; LORO cannot hold out whole runs.")

n_norm = sum(1 for f in target_files if parse_clip(f)["condition"] in TGT["normal_conditions"])
print(f"✅ Target domain verified: {TARGET_ROOT}")
print(f"   {len(target_files)} clips = {n_norm} normal + {len(target_files) - n_norm} abnormal, "
      f"{len(grid)} condition combinations × {TGT['n_runs']} runs, grid complete")

## 5 · Load Stage 0 outputs

Read-only. Three things:

1. **`stage0_splits_seed42.json`** — data splits. **This stage never re-splits**;
   small shift uses exactly the 160 normal + 320 abnormal fixed for each machine ID in Stage 0.
2. **`stage0_clip_scores_<model>.csv`** — per-clip anomaly scores. Its 160 rows with `label==0`
   are the model's reconstruction MSE on **the normal samples of its own machine ID's eval set**; their median is the **denominator of the threshold shift ratio**,
   so no forward pass needs to be rerun.
3. **`stage0_source_baseline.csv`** — source-domain AUC / best F1, the reference for "degradation magnitude".

In [ ]:
SPLITS_PATH = os.path.join(BASELINE_LOGS, f"stage0_splits_seed{SEED}.json")
if not os.path.exists(SPLITS_PATH):
    raise FileNotFoundError(
        f"❌ Stage 0 split manifest not found: {SPLITS_PATH}\n"
        f"   Actual contents of {BASELINE_LOGS}:\n     "
        + "\n     ".join(sorted(os.listdir(BASELINE_LOGS)) if os.path.isdir(BASELINE_LOGS) else ["(directory does not exist)"])
        + "\n   → If the path is wrong, edit BASELINE_DIR in section 2."
    )

with open(SPLITS_PATH) as f:
    blob = json.load(f)

if blob.get("config_sha256") == CONFIG_SHA256:
    print(f"✅ Split manifest config sha256 matches the config embedded in this notebook")
else:
    print(f"⚠️ config sha256 mismatch: Stage 0 used {str(blob.get('config_sha256'))[:16]}..., "
          f"this notebook embeds {CONFIG_SHA256[:16]}...")
    print(f"   Splits are read from file, not recomputed, so correctness is unaffected; it only means config.yaml changed between stages.")

assert blob["seed"] == SEED, f"Split manifest seed is {blob['seed']}, does not match config seed {SEED}"

# relative paths → absolute paths in the current runtime
splits = {name: {k: [os.path.join(MIMII_ROOT, p) for p in v] for k, v in d.items()}
          for name, d in blob["splits"].items()}

for name in ALL_MODELS:
    s = splits[name]
    assert len(s["eval_normal"]) == cfg["source_domain"]["eval_set"]["n_normal"]
    assert len(s["eval_abnormal"]) == cfg["source_domain"]["eval_set"]["n_abnormal"]
    missing = [p for p in s["eval_normal"] + s["eval_abnormal"] if not os.path.exists(p)]
    assert not missing, f"{name} eval set: {len(missing)} files not found on local disk, e.g. {missing[0]}"

print(f"\n{'Model':<10}{'Train pool':>11}{'Eval normal':>12}{'Eval abnormal':>14}")
print("-" * 47)
for name in ALL_MODELS:
    s = splits[name]
    print(f"{name:<10}{len(s['train_pool']):>11}{len(s['eval_normal']):>12}{len(s['eval_abnormal']):>14}")

# --- Source-domain baseline: AUC / best F1 ---
base_df = pd.read_csv(os.path.join(BASELINE_LOGS, "stage0_source_baseline.csv")).set_index("model")
SOURCE_AUC = base_df["auc"].to_dict()
SOURCE_F1  = base_df["best_f1"].to_dict()

# --- Threshold shift ratio denominator: median reconstruction MSE on each model's own-ID eval normals ---
SOURCE_MSE_MEDIAN = {}
for name in ALL_MODELS:
    p = os.path.join(BASELINE_LOGS, f"stage0_clip_scores_{name}.csv")
    d = pd.read_csv(p)
    normals = d.loc[d["label"] == 0, "score"]
    assert len(normals) == cfg["source_domain"]["eval_set"]["n_normal"], \
        f"{p} has {len(normals)} normal rows, expected {cfg['source_domain']['eval_set']['n_normal']}"
    SOURCE_MSE_MEDIAN[name] = float(normals.median())

print(f"\n{'Model':<10}{'Src AUC':>10}{'Src best F1':>12}{'Src normal MSE med':>20}")
print("-" * 54)
for name in ALL_MODELS:
    print(f"{name:<10}{SOURCE_AUC[name]:>10.4f}{SOURCE_F1[name]:>12.4f}{SOURCE_MSE_MEDIAN[name]:>20.6f}")
print("\n(The last column is the threshold shift ratio denominator, taken from Stage 0 per-clip scores; no forward pass rerun)")

## 6 · Feature extraction and caching

Same cache directory and same extraction logic as Stage 0 (`librosa.load(sr=None, mono=True)` → 64 mel →
`20/power·log10` → 5 frames concatenated into 320 dims).

- **MIMII**: if the Stage 0 cache is still present (same runtime, not restarted), it is reused instantly;
  the cached file list only needs to be a **superset** of what this stage requires. Otherwise it is rebuilt as the same full set as Stage 0
  (all normals + each ID's 320 eval abnormals), so Stage 2 can keep reusing it without extracting again.
- **Target domain**: all 180 clips are freshly extracted (~71 MB, 20–60 s).

The cache lives on local disk; **after a runtime restart, rerun this cell to rebuild it**.

In [ ]:
os.makedirs(FEATURE_DIR, exist_ok=True)


def file_to_vector_array(file_name):
    """Line-for-line identical to baseline_runner.ipynb / Stage 0."""
    n_mels, frames = FEAT["n_mels"], FEAT["frames"]
    n_fft, hop_length, power = FEAT["n_fft"], FEAT["hop_length"], FEAT["power"]

    y, sr = librosa.load(file_name, sr=None, mono=True)
    mel_spectrogram = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power
    )
    log_mel_spectrogram = 20.0 / power * np.log10(mel_spectrogram + sys.float_info.epsilon)

    vectorarray_size = log_mel_spectrogram.shape[1] - frames + 1
    if vectorarray_size < 1:
        return np.empty((0, n_mels * frames), float)
    dims = n_mels * frames
    vectorarray = np.zeros((vectorarray_size, dims), float)
    for t in range(frames):
        vectorarray[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vectorarray_size].T
    return vectorarray


def build_cache(key, files, base, desc):
    """Extract and cache. Reuse the existing cache if its file list is a superset of the required set."""
    stem = os.path.join(FEATURE_DIR, key)
    npy_path, meta_path = stem + ".npy", stem + ".meta.json"
    rel = [os.path.relpath(f, base) for f in files]

    if os.path.exists(npy_path) and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if set(rel) <= set(meta["files"]):
            return npy_path, meta

    chunks, offsets = [], [0]
    for f in tqdm(files, desc=desc, unit="clip", leave=False):
        v = file_to_vector_array(f).astype(np.float32)
        chunks.append(v)
        offsets.append(offsets[-1] + len(v))
    X = np.concatenate(chunks, axis=0)
    del chunks
    np.save(npy_path, X)
    meta = {"files": rel, "offsets": offsets, "n_clips": len(files),
            "n_vectors": int(X.shape[0]), "dim": int(X.shape[1])}
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    del X
    return npy_path, meta


t0 = time.time()
CACHE, CACHE_BASE = {}, {}

# ---- MIMII: same full set as Stage 0 ----
print("MIMII features:")
for mid in MACHINE_IDS:
    need_normal = sorted(set(splits[mid]["train_pool"]) | set(splits[mid]["eval_normal"]))
    for lab, files in (("normal", need_normal), ("abnormal", splits[mid]["eval_abnormal"])):
        key = f"mimii_{mid}_{lab}"
        existed = os.path.exists(os.path.join(FEATURE_DIR, key + ".npy"))
        CACHE[key] = build_cache(key, files, MIMII_ROOT, f"{mid}/{lab}")
        CACHE_BASE[key] = MIMII_ROOT
        print(f"  {key:<24} {len(files):>5} clips  {'cached' if existed else 'extracted'}")

# ---- Target domain: 180 clips ----
print("Target-domain features:")
existed = os.path.exists(os.path.join(FEATURE_DIR, "target_all.npy"))
CACHE["target_all"] = build_cache("target_all", target_files, TARGET_ROOT, "target")
CACHE_BASE["target_all"] = TARGET_ROOT
print(f"  {'target_all':<24} {len(target_files):>5} clips  {'cached' if existed else 'extracted'}")

per_clip = set()
for key in CACHE:
    off = CACHE[key][1]["offsets"]
    per_clip |= {off[i + 1] - off[i] for i in range(len(off) - 1)}
print(f"\nSet of vectors-per-clip values: {sorted(per_clip)}  (expected 309 for 10s@16kHz)")

cache_gb = sum(os.path.getsize(os.path.join(FEATURE_DIR, f)) for f in os.listdir(FEATURE_DIR)) / 1024**3
print(f"✅ Features ready: {FEATURE_DIR}  ({cache_gb:.2f} GB), cell took {time.time() - t0:.0f} s")

## 7 · Load the 5 frozen models + evaluation functions

Models are loaded from `Baseline/models/` and immediately set to `eval()` and `requires_grad_(False)`;
there is no backpropagation anywhere in this stage.

**Per-clip anomaly score** = mean reconstruction MSE over all frames of the clip (identical to Stage 0 and the baseline).
**Best F1** = maximum over 1000 thresholds scanned across `[min, max]`.

In [ ]:
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, 8),         nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 64),         nn.ReLU(),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, input_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


MODELS = {}
for name in ALL_MODELS:
    p = os.path.join(BASELINE_MODELS, f"baseline_fan_{name}.pth")
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"❌ Stage 0 weights not found: {p}\n   Actual contents of {BASELINE_MODELS}:\n     "
            + "\n     ".join(sorted(os.listdir(BASELINE_MODELS)) if os.path.isdir(BASELINE_MODELS) else ["(directory does not exist)"])
            + "\n   → If the path is wrong, edit BASELINE_DIR in section 2.")
    ck = torch.load(p, map_location=DEVICE, weights_only=False)
    m = MIMII_Baseline_AE(ck.get("input_dim", FEAT["input_dim"])).to(DEVICE)
    m.load_state_dict(ck["model_state_dict"])
    m.eval().requires_grad_(False)          # frozen: no training in this stage
    MODELS[name] = m
    print(f"  ✅ {name:<8} loaded | Stage 0 recorded AUC={ck.get('auc', float('nan')):.4f}")

assert all(not p.requires_grad for m in MODELS.values() for p in m.parameters()), "models not frozen"


def load_vectors(key, files):
    """Fetch vectors for the given files from cache; return (X, per-clip slice ranges)."""
    npy_path, meta = CACHE[key]
    base = CACHE_BASE[key]
    X = np.load(npy_path, mmap_mode="r")
    index = {f: i for i, f in enumerate(meta["files"])}
    off = meta["offsets"]
    parts, slices, cur = [], [], 0
    for f in files:
        i = index[os.path.relpath(f, base)]
        seg = np.asarray(X[off[i]:off[i + 1]])
        parts.append(seg)
        slices.append((cur, cur + len(seg)))
        cur += len(seg)
    return np.concatenate(parts, axis=0), slices


@torch.no_grad()
def clip_scores(model, X, slices):
    Xt = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
    out = np.empty(len(slices), dtype=np.float64)
    for k, (a, b) in enumerate(slices):
        seg = Xt[a:b]
        out[k] = torch.mean((seg - model(seg)) ** 2, dim=1).mean().double().cpu().item()
    del Xt
    return out


def find_best_f1(y_true, y_scores, n_points=None):
    n_points = N_SCAN if n_points is None else n_points
    best = {"f1": 0.0, "threshold": float("nan"), "precision": 0.0, "recall": 0.0, "accuracy": 0.0}
    for t in np.linspace(min(y_scores), max(y_scores), n_points):
        pred = (y_scores > t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"f1": float(f1), "threshold": float(t), "precision": float(p),
                    "recall": float(r), "accuracy": float(accuracy_score(y_true, pred))}
    return best


print(f"\n✅ {len(MODELS)} models loaded and frozen")

## 8 · Small-shift evaluation (12 pairs)

4 single-ID models × their 3 **unseen** machine IDs each. The merged model is excluded — it saw all four IDs during pretraining,
so "small shift" does not exist for it.

The eval set is **the target ID's** 160 normal + 320 abnormal **fixed in Stage 0**, the same files that ID's own model
was evaluated on in Stage 0, so "start model on id_02" can be compared directly with "id_02 model on id_02".

Threshold shift ratio = median MSE of the target ID's 160 normals ÷ median MSE of the start model's own 160 normals (denominator from Stage 0).

In [ ]:
small_rows, small_scores = [], []

pairs = [(s, t) for s in MACHINE_IDS for t in MACHINE_IDS if s != t]
assert len(pairs) == S1["small_shift_pairs"] == 12

for i, (src_model, tgt_id) in enumerate(pairs, 1):
    s = splits[tgt_id]
    Xn, sn = load_vectors(f"mimii_{tgt_id}_normal",   s["eval_normal"])
    Xa, sa = load_vectors(f"mimii_{tgt_id}_abnormal", s["eval_abnormal"])

    sc_n = clip_scores(MODELS[src_model], Xn, sn)
    sc_a = clip_scores(MODELS[src_model], Xa, sa)
    scores = np.concatenate([sc_n, sc_a])
    labels = np.array([0] * len(sn) + [1] * len(sa))
    files  = s["eval_normal"] + s["eval_abnormal"]

    auc = float(metrics.roc_auc_score(labels, scores))
    best = find_best_f1(labels, scores)
    tgt_median = float(np.median(sc_n))
    src_median = SOURCE_MSE_MEDIAN[src_model]

    small_rows.append({
        "source_model": src_model, "target_id": tgt_id,
        "source_auc": SOURCE_AUC[src_model], "auc": auc,
        "auc_drop": SOURCE_AUC[src_model] - auc,
        "source_best_f1": SOURCE_F1[src_model], "best_f1": best["f1"],
        "best_threshold": best["threshold"], "precision": best["precision"], "recall": best["recall"],
        "source_normal_mse_median": src_median,
        "target_normal_mse_median": tgt_median,
        "shift_ratio": tgt_median / src_median,
        "n_eval_clips": len(labels),
    })
    for f, y, sc in zip(files, labels, scores):
        small_scores.append({"source_model": src_model, "target_id": tgt_id,
                             "file": os.path.relpath(f, MIMII_ROOT), "label": int(y), "score": sc})

    print(f"  [{i:>2}/12] {src_model} → {tgt_id} | AUC {auc:.4f} (source {SOURCE_AUC[src_model]:.4f}, "
          f"drop {SOURCE_AUC[src_model] - auc:+.4f}) | best F1 {best['f1']:.4f} | shift ratio {tgt_median / src_median:.2f}×")

small_df = pd.DataFrame(small_rows)
small_df.to_csv(os.path.join(STAGE1_LOGS, "stage1_small_shift.csv"), index=False)
pd.DataFrame(small_scores).to_csv(os.path.join(STAGE1_LOGS, "stage1_clip_scores_small.csv"), index=False)
print(f"\n✅ Small shift: 12 pairs done, written to {STAGE1_LOGS}")

## 9 · Large-shift evaluation (5 models × 10-fold LORO)

**Fold split**: fold k (0-based) holds out the whole run `run == k+1`, i.e. 18 clips (6 normal + 12 abnormal).

**Two aggregation schemes** (`proposal/proposal.md` §6.2):

- **Pooled** (headline number): scores of all 180 clips together give one AUC and one best F1
- **Per-fold**: computed per fold, reported as mean ± std over the 10 folds

The models are frozen, so each clip is forwarded only once; the fold split is just a post-hoc partition (see the note at the top).

**The numerator of the threshold shift ratio uses all 60 normal clips**, not per fold — frozen evaluation has no leakage risk,
and the median over 60 clips is far more stable than over the 6 clips of a single fold.

In [ ]:
# target domain: score all 180 clips in one pass
target_meta = [parse_clip(f) for f in target_files]
target_labels = np.array([0 if d["condition"] in TGT["normal_conditions"] else 1 for d in target_meta])
target_runs = np.array([d["run"] for d in target_meta])
Xt, st = load_vectors("target_all", target_files)

large_rows, fold_rows, large_scores = [], [], []

for i, name in enumerate(ALL_MODELS, 1):
    scores = clip_scores(MODELS[name], Xt, st)

    # --- pooled (headline number) ---
    pooled_auc = float(metrics.roc_auc_score(target_labels, scores))
    pooled_best = find_best_f1(target_labels, scores)

    # --- per-fold ---
    fold_auc, fold_f1 = [], []
    for k in range(LORO["n_folds"]):
        sel = target_runs == (k + 1)
        assert sel.sum() == LORO["clips_per_fold"], f"fold {k} has {sel.sum()} clips, expected {LORO['clips_per_fold']}"
        y, sc = target_labels[sel], scores[sel]
        assert (y == 0).sum() == LORO["normals_per_fold"] and (y == 1).sum() == LORO["abnormals_per_fold"]
        a = float(metrics.roc_auc_score(y, sc))
        b = find_best_f1(y, sc)
        fold_auc.append(a)
        fold_f1.append(b["f1"])
        fold_rows.append({"model": name, "fold": k, "held_out_run": k + 1,
                          "n_clips": int(sel.sum()), "auc": a, "best_f1": b["f1"]})

    # --- threshold shift ratio: numerator = median MSE over all 60 normal clips ---
    tgt_median = float(np.median(scores[target_labels == 0]))
    src_median = SOURCE_MSE_MEDIAN[name]

    large_rows.append({
        "model": name,
        "source_auc": SOURCE_AUC[name], "pooled_auc": pooled_auc,
        "auc_drop": SOURCE_AUC[name] - pooled_auc,
        "source_best_f1": SOURCE_F1[name], "pooled_best_f1": pooled_best["f1"],
        "pooled_threshold": pooled_best["threshold"],
        "fold_auc_mean": float(np.mean(fold_auc)), "fold_auc_std": float(np.std(fold_auc)),
        "fold_f1_mean": float(np.mean(fold_f1)),   "fold_f1_std": float(np.std(fold_f1)),
        "source_normal_mse_median": src_median,
        "target_normal_mse_median": tgt_median,
        "shift_ratio": tgt_median / src_median,
        "n_eval_clips": len(target_labels),
    })
    for f, d, y, sc in zip(target_files, target_meta, target_labels, scores):
        large_scores.append({"model": name, "file": os.path.basename(f), "condition": d["condition"],
                             "voltage": d["voltage"], "noise": d["noise"], "run": d["run"],
                             "label": int(y), "score": sc})

    print(f"  [{i}/5] {name:<8} | pooled AUC {pooled_auc:.4f} (source {SOURCE_AUC[name]:.4f}, "
          f"drop {SOURCE_AUC[name] - pooled_auc:+.4f}) | pooled F1 {pooled_best['f1']:.4f}")
    print(f"           per-fold AUC {np.mean(fold_auc):.4f} ± {np.std(fold_auc):.4f} | "
          f"per-fold F1 {np.mean(fold_f1):.4f} ± {np.std(fold_f1):.4f} | shift ratio {tgt_median / src_median:.2f}×")

large_df = pd.DataFrame(large_rows)
large_df.to_csv(os.path.join(STAGE1_LOGS, "stage1_large_shift.csv"), index=False)
pd.DataFrame(fold_rows).to_csv(os.path.join(STAGE1_LOGS, "stage1_large_shift_folds.csv"), index=False)
pd.DataFrame(large_scores).to_csv(os.path.join(STAGE1_LOGS, "stage1_clip_scores_large.csv"), index=False)
print(f"\n✅ Large shift: 5 models × {LORO['n_folds']} folds done, written to {STAGE1_LOGS}")

## 10 · Summary

Three tables + one line with the core RQ1 comparison.

**Reading note**: the trivial floor of `best F1` is **0.800** (obtained by labeling everything abnormal at a 1:2 class ratio),
so F1 indicates real discriminative power only when clearly above 0.80; **AUC is primary**.
A LORO fold has only 18 clips (6 normal + 12 abnormal), so single-fold metrics are inherently high-variance and a large std is normal —
which is exactly why both aggregation schemes are reported.

In [ ]:
print("Table 1 · Small shift (12 pairs, between MIMII machine IDs)")
print("=" * 94)
print(f"{'Start':<8}{'Target':<8}{'Src AUC':>9}{'Tgt AUC':>9}{'AUC drop':>9}{'Best F1':>9}"
      f"{'Src MSE med':>12}{'Tgt MSE med':>13}{'Shift ratio':>12}")
print("-" * 94)
for r in small_rows:
    print(f"{r['source_model']:<8}{r['target_id']:<8}{r['source_auc']:>9.4f}{r['auc']:>9.4f}"
          f"{r['auc_drop']:>9.4f}{r['best_f1']:>9.4f}{r['source_normal_mse_median']:>12.4f}"
          f"{r['target_normal_mse_median']:>13.4f}{r['shift_ratio']:>11.2f}×")
print("=" * 94)
print(f"{'Mean':<16}{small_df['source_auc'].mean():>9.4f}{small_df['auc'].mean():>9.4f}"
      f"{small_df['auc_drop'].mean():>9.4f}{small_df['best_f1'].mean():>9.4f}"
      f"{'':>12}{'':>13}{small_df['shift_ratio'].mean():>11.2f}×")

print("\n\nTable 2 · Large shift (5 models → self-recorded fan; pooled is the headline number)")
print("=" * 96)
print(f"{'Model':<10}{'Src AUC':>9}{'Pool AUC':>9}{'AUC drop':>9}{'Pool F1':>9}"
      f"{'Fold AUC (mean±sd)':>20}{'Fold F1 (mean±sd)':>19}{'Shift ratio':>12}")
print("-" * 96)
for r in large_rows:
    print(f"{r['model']:<10}{r['source_auc']:>9.4f}{r['pooled_auc']:>9.4f}{r['auc_drop']:>9.4f}"
          f"{r['pooled_best_f1']:>9.4f}"
          f"{r['fold_auc_mean']:>13.4f}±{r['fold_auc_std']:<6.4f}"
          f"{r['fold_f1_mean']:>12.4f}±{r['fold_f1_std']:<6.4f}{r['shift_ratio']:>11.2f}×")
print("=" * 96)
print(f"{'Mean':<10}{large_df['source_auc'].mean():>9.4f}{large_df['pooled_auc'].mean():>9.4f}"
      f"{large_df['auc_drop'].mean():>9.4f}{large_df['pooled_best_f1'].mean():>9.4f}"
      f"{'':>20}{'':>19}{large_df['shift_ratio'].mean():>11.2f}×")

print("\n\nTable 3 · RQ1 core comparison: small shift vs large shift")
print("=" * 62)
print(f"{'':<16}{'Mean AUC':>11}{'Mean AUC drop':>15}{'Mean shift ratio':>18}")
print("-" * 62)
print(f"{'Small (12 pairs)':<16}{small_df['auc'].mean():>11.4f}"
      f"{small_df['auc_drop'].mean():>15.4f}{small_df['shift_ratio'].mean():>17.2f}×")
print(f"{'Large (5 models)':<16}{large_df['pooled_auc'].mean():>11.4f}"
      f"{large_df['auc_drop'].mean():>15.4f}{large_df['shift_ratio'].mean():>17.2f}×")
print("=" * 62)
_sd = small_df["auc_drop"].mean()
print("Degradation ratio (large/small): AUC drop "
      + (f"{large_df['auc_drop'].mean() / _sd:.2f}×" if _sd > 0.01 else "small shift shows almost no degradation; ratio not meaningful")
      + f"  |  threshold shift {large_df['shift_ratio'].mean() / small_df['shift_ratio'].mean():.2f}×")
print(f"\n(The proposal's opening hypothesis puts the large-shift threshold shift at roughly 5-10x; compare with the table above)")

summary_path = os.path.join(STAGE1_LOGS, "stage1_summary.json")
with open(summary_path, "w") as f:
    json.dump({
        "config_sha256": CONFIG_SHA256,
        "small_shift": {"n_pairs": len(small_rows),
                        "mean_auc": float(small_df["auc"].mean()),
                        "mean_auc_drop": float(small_df["auc_drop"].mean()),
                        "mean_shift_ratio": float(small_df["shift_ratio"].mean())},
        "large_shift": {"n_models": len(large_rows),
                        "mean_pooled_auc": float(large_df["pooled_auc"].mean()),
                        "mean_auc_drop": float(large_df["auc_drop"].mean()),
                        "mean_shift_ratio": float(large_df["shift_ratio"].mean())},
    }, f, indent=2)

print(f"\n✅ Stage 1 done. Outputs in {STAGE1_LOGS}/:")
for fn in ["stage1_small_shift.csv        ← Table 1 (12 rows)",
           "stage1_large_shift.csv        ← Table 2 (5 rows)",
           "stage1_large_shift_folds.csv  ← per-fold details (5×10 = 50 rows)",
           "stage1_clip_scores_small.csv  ← small-shift per-clip scores (12×480 = 5760 rows)",
           "stage1_clip_scores_large.csv  ← large-shift per-clip scores (5×180 = 900 rows)",
           "stage1_summary.json           ← RQ1 summary"]:
    print(f"   {fn}")
large_df